# Занятие 10. Теория: как учится нейронная сеть

План:

1. сеть — функция трех аргументов: входа, весов и правильного ответа;
2. вычислительный граф и цепное правило на числах;
3. нейрон с сигмоидой: прямой и обратный проход по шагам;
4. функции активации: зачем нужны и чем отличаются;
5. софтмакс: от одного класса к десяти;
6. прямой проход по слоям: как картинка превращается в десять вероятностей;
7. инициализация весов: почему не одним числом и не N(0, 1);
8. сеть для MNIST: обучение;
9. производные по входу: атака на сеть и карта важности пикселей;
10. производные по правильному ответу.

Код, который только рисует, лежит в файле `sem10_utils.py` рядом с ноутбуком.

In [ ]:
import warnings

import numpy as np
import pandas as pd

from sklearn.datasets import fetch_openml, make_moons
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier

from sem10_utils import (
    animate_forward_pass, animate_graph, draw_graph, plot_activations, plot_architecture, plot_curve,
    plot_decision_maps, plot_histories, plot_init_comparison, plot_network_function, plot_softmax_bars,
    show_adversarial, show_digits, show_saliency_pairs, show_table,
)

warnings.filterwarnings("ignore")
SEED = 42

## 1. Сеть — функция трех аргументов

Сеть получает вход $x$ — картинку из 784 пикселей — и выдает $\hat{y} = f(x; W)$: вероятности десяти
цифр. Что именно она считает, задают веса $W$ — около 100 000 чисел. Потеря сравнивает выход
с правильным ответом $y$:

$$L(x, W, y) = L\big(f(x;\, W),\, y\big).$$

Это обычная функция трех аргументов, и у нее есть производные по каждому из них:

- $\frac{\partial L}{\partial W}$ — как изменить веса, чтобы ошибка уменьшилась: **обучение**;
- $\frac{\partial L}{\partial x}$ — как изменить картинку, чтобы ошибка выросла или уменьшилась: атаки на сеть,
  карты важности пикселей (раздел 9);
- $\frac{\partial L}{\partial y}$ — как изменить сам ответ: обучение меток (раздел 10).

Обучение — градиентный спуск по весам: $W \leftarrow W - \eta\, \frac{\partial L}{\partial W}$. Весов сто тысяч, и считать производную
по каждому отдельно слишком долго. **Обратное распространение ошибки** (backpropagation) считает
все производные за один проход от потери к входу.

In [ ]:
plot_network_function()

## 2. Вычислительный граф

Любое вычисление можно записать графом простых операций. Пример: $f(x, y, z) = (x + y)\,z$
при $x = -2$, $y = 5$, $z = -4$. Промежуточная величина $q = x + y = 3$, ответ $f = q z = -12$.

**Прямой проход** — слева направо: считаем значения на ребрах (зеленые числа).

**Обратный проход** — справа налево: на каждом ребре пишем производную ответа $f$ по величине
на этом ребре (красные числа). Начинаем с $\frac{\partial f}{\partial f} = 1$. У узла $f = q z$ локальные производные
$\frac{\partial f}{\partial q} = z = -4$ и $\frac{\partial f}{\partial z} = q = 3$. У узла $q = x + y$ локальные производные $\frac{\partial q}{\partial x} = \frac{\partial q}{\partial y} = 1$,
и по **цепному правилу**

$$\frac{\partial f}{\partial x} = \frac{\partial f}{\partial q}\cdot\frac{\partial q}{\partial x} = -4 \cdot 1 = -4, \qquad \frac{\partial f}{\partial y} = \frac{\partial f}{\partial q}\cdot\frac{\partial q}{\partial y} = -4.$$

Каждый узел знает только свою локальную производную и умножает на нее то, что пришло справа.

Граф ниже интерактивный: слайдер или кнопки переключают шаги. Сначала слева направо появляются
зеленые значения прямого прохода, затем справа налево — красные градиенты обратного. У каждого
ребра есть имя: на прямом проходе подпись $q = 3$ — значение величины на ребре, на обратном
$\partial f / \partial q = -4$ — производная ответа $f$ по этой величине. Справа — как получено текущее число:
полная цепочка производных от $\partial f/\partial f$ до нужного ребра и откуда взят каждый множитель.

In [ ]:
animate_graph(
    nodes={
        "x": (0.0, 2.6, "x", "input"), "y": (0.0, 0.8, "y", "input"), "z": (0.0, -1.0, "z", "input"),
        "plus": (3.0, 1.7, "+", "op"), "times": (6.2, 0.4, "×", "op"), "f": (8.8, 0.4, "f", "input"),
    },
    edges=[
        ("x", "plus", -2, -4, "x"), ("y", "plus", 5, -4, "y"), ("plus", "times", 3, -4, "q"),
        ("z", "times", -4, 3, "z"), ("times", "f", -12, 1, "f"),
    ],
    explanations=[
        ('x', '$x = -2$ — входная переменная, значение задано.', '$q(x, y) = x + y \\Rightarrow \\frac{\\partial q}{\\partial x} = 1$\n$\\frac{\\partial f}{\\partial x} = \\frac{\\partial f}{\\partial f} \\cdot \\frac{\\partial f}{\\partial q} \\cdot \\frac{\\partial q}{\\partial x} = \\frac{\\partial f}{\\partial q} \\cdot \\frac{\\partial q}{\\partial x} = (-4) \\cdot 1 = -4$'),
        ('y', '$y = 5$ — входная переменная, значение задано.', '$q(x, y) = x + y \\Rightarrow \\frac{\\partial q}{\\partial y} = 1$\n$\\frac{\\partial f}{\\partial y} = \\frac{\\partial f}{\\partial f} \\cdot \\frac{\\partial f}{\\partial q} \\cdot \\frac{\\partial q}{\\partial y} = \\frac{\\partial f}{\\partial q} \\cdot \\frac{\\partial q}{\\partial y} = (-4) \\cdot 1 = -4$'),
        ('q', '$q(x, y) = x + y = -2 + 5 = 3$', '$f(q, z) = q \\cdot z \\Rightarrow \\frac{\\partial f}{\\partial q} = z = -4$\n$\\frac{\\partial f}{\\partial q} = \\frac{\\partial f}{\\partial f} \\cdot \\frac{\\partial f}{\\partial q} = 1 \\cdot (-4) = -4$'),
        ('z', '$z = -4$ — входная переменная, значение задано.', '$f(q, z) = q \\cdot z \\Rightarrow \\frac{\\partial f}{\\partial z} = q = 3$\n$\\frac{\\partial f}{\\partial z} = \\frac{\\partial f}{\\partial f} \\cdot \\frac{\\partial f}{\\partial z} = 1 \\cdot 3 = 3$'),
        ('f', '$f(q, z) = q \\cdot z = 3 \\cdot (-4) = -12$', 'Начало обратного прохода:\n$\\frac{\\partial f}{\\partial f} = 1$ — производная $f$ по самой себе.'),
    
    ],
    title=r"$f(x, y, z) = (x + y) \cdot z$:   $q(x, y) = x + y$,   $f(q, z) = q \cdot z$",
    size=(18, 5.6),
)

Смысл красных чисел: если увеличить $x$ на малое $\delta$, ответ изменится на $-4\delta$. Проверка:
$f(-2 + 0.01,\ 5,\ -4) = (3.01) \cdot (-4) = -12.04$ — меньше на $0.04 = 4 \cdot 0.01$.

## 3. Нейрон с сигмоидой

Нейрон с двумя входами: $f(w, x) = \sigma(w_0 x_0 + w_1 x_1 + w_2)$, $\sigma(t) = \frac{1}{1 + e^{-t}}$. Значения:
$w_0 = 2$, $x_0 = -1$, $w_1 = -3$, $x_1 = -2$, $w_2 = -3$. Сигмоиду разложим на простые операции:
умножение на $-1$, экспоненту, прибавление 1 и $\frac{1}{t}$. Прямой проход:

$$w_0 x_0 + w_1 x_1 + w_2 = -2 + 6 - 3 = 1, \quad e^{-1} = 0.37, \quad 1 + 0.37 = 1.37, \quad \frac{1}{1.37} = 0.73.$$

Обратный проход, справа налево, локальные производные:

- $\frac{1}{t}$: производная $-\frac{1}{t^2} = -\frac{1}{1.37^2} = -0.53$;
- $t + 1$: производная 1, градиент не меняется, $-0.53$;
- $e^{t}$: производная $e^{t} = e^{-1} = 0.37$, градиент $-0.53 \cdot 0.37 = -0.20$;
- $\cdot(-1)$: градиент $0.20$;
- сумма передает $0.20$ каждому слагаемому, и $\frac{\partial f}{\partial w_2} = 0.20$;
- произведение $w_0 x_0$: $\frac{\partial f}{\partial w_0} = x_0 \cdot 0.20 = -0.20$, $\frac{\partial f}{\partial x_0} = w_0 \cdot 0.20 = 0.40$;
  аналогично $\frac{\partial f}{\partial w_1} = x_1 \cdot 0.20 = -0.40$, $\frac{\partial f}{\partial x_1} = w_1 \cdot 0.20 = -0.60$.

In [ ]:
animate_graph(
    nodes={
        "w0": (0, 6.2, "w₀", "input"), "x0": (0, 4.2, "x₀", "input"), "w1": (0, 2.4, "w₁", "input"),
        "x1": (0, 0.4, "x₁", "input"), "w2": (0, -1.6, "w₂", "input"),
        "m0": (3.4, 5.2, "×", "op"), "m1": (3.4, 1.4, "×", "op"), "s0": (6.4, 3.3, "+", "op"),
        "s1": (9.4, 1.4, "+", "op"), "neg": (12.4, 1.4, "·(−1)", "op"), "exp": (15.4, 1.4, "exp", "op"),
        "add": (18.4, 1.4, "+1", "op"), "inv": (21.4, 1.4, "1/t", "op"), "f": (24.0, 1.4, "f", "input"),
    },
    edges=[
        ("w0", "m0", 2, -0.2, "w₀"), ("x0", "m0", -1, 0.4, "x₀"), ("w1", "m1", -3, -0.4, "w₁"), ("x1", "m1", -2, -0.6, "x₁"),
        ("m0", "s0", -2, 0.2, "a"), ("m1", "s0", 6, 0.2, "b"), ("s0", "s1", 4, 0.2, "c"), ("w2", "s1", -3, 0.2, "w₂"),
        ("s1", "neg", 1, 0.2, "s"), ("neg", "exp", -1, -0.2, "u"), ("exp", "add", 0.37, -0.53, "v"),
        ("add", "inv", 1.37, -0.53, "t"), ("inv", "f", 0.73, 1, "f"),
    ],
    explanations=[
        ('w₀', '$w_0 = 2$ — вес, значение задано.', '$a(w_0, x_0) = w_0 \\cdot x_0 \\Rightarrow \\frac{\\partial a}{\\partial w_0} = x_0 = -1$\n$\\frac{\\partial f}{\\partial w_0} = \\frac{\\partial f}{\\partial f} \\cdot \\frac{\\partial f}{\\partial t} \\cdot \\frac{\\partial t}{\\partial v} \\cdot \\frac{\\partial v}{\\partial u} \\cdot \\frac{\\partial u}{\\partial s} \\cdot \\frac{\\partial s}{\\partial c} \\cdot \\frac{\\partial c}{\\partial a} \\cdot \\frac{\\partial a}{\\partial w_0} = \\frac{\\partial f}{\\partial a} \\cdot \\frac{\\partial a}{\\partial w_0} = 0.2 \\cdot (-1) = -0.2$'),
        ('x₀', '$x_0 = -1$ — вход, значение задано.', '$a(w_0, x_0) = w_0 \\cdot x_0 \\Rightarrow \\frac{\\partial a}{\\partial x_0} = w_0 = 2$\n$\\frac{\\partial f}{\\partial x_0} = \\frac{\\partial f}{\\partial f} \\cdot \\frac{\\partial f}{\\partial t} \\cdot \\frac{\\partial t}{\\partial v} \\cdot \\frac{\\partial v}{\\partial u} \\cdot \\frac{\\partial u}{\\partial s} \\cdot \\frac{\\partial s}{\\partial c} \\cdot \\frac{\\partial c}{\\partial a} \\cdot \\frac{\\partial a}{\\partial x_0} = \\frac{\\partial f}{\\partial a} \\cdot \\frac{\\partial a}{\\partial x_0} = 0.2 \\cdot 2 = 0.4$'),
        ('w₁', '$w_1 = -3$ — вес, значение задано.', '$b(w_1, x_1) = w_1 \\cdot x_1 \\Rightarrow \\frac{\\partial b}{\\partial w_1} = x_1 = -2$\n$\\frac{\\partial f}{\\partial w_1} = \\frac{\\partial f}{\\partial f} \\cdot \\frac{\\partial f}{\\partial t} \\cdot \\frac{\\partial t}{\\partial v} \\cdot \\frac{\\partial v}{\\partial u} \\cdot \\frac{\\partial u}{\\partial s} \\cdot \\frac{\\partial s}{\\partial c} \\cdot \\frac{\\partial c}{\\partial b} \\cdot \\frac{\\partial b}{\\partial w_1} = \\frac{\\partial f}{\\partial b} \\cdot \\frac{\\partial b}{\\partial w_1} = 0.2 \\cdot (-2) = -0.4$'),
        ('x₁', '$x_1 = -2$ — вход, значение задано.', '$b(w_1, x_1) = w_1 \\cdot x_1 \\Rightarrow \\frac{\\partial b}{\\partial x_1} = w_1 = -3$\n$\\frac{\\partial f}{\\partial x_1} = \\frac{\\partial f}{\\partial f} \\cdot \\frac{\\partial f}{\\partial t} \\cdot \\frac{\\partial t}{\\partial v} \\cdot \\frac{\\partial v}{\\partial u} \\cdot \\frac{\\partial u}{\\partial s} \\cdot \\frac{\\partial s}{\\partial c} \\cdot \\frac{\\partial c}{\\partial b} \\cdot \\frac{\\partial b}{\\partial x_1} = \\frac{\\partial f}{\\partial b} \\cdot \\frac{\\partial b}{\\partial x_1} = 0.2 \\cdot (-3) = -0.6$'),
        ('a', '$a(w_0, x_0) = w_0 \\cdot x_0 = 2 \\cdot (-1) = -2$', '$c(a, b) = a + b \\Rightarrow \\frac{\\partial c}{\\partial a} = 1$\n$\\frac{\\partial f}{\\partial a} = \\frac{\\partial f}{\\partial f} \\cdot \\frac{\\partial f}{\\partial t} \\cdot \\frac{\\partial t}{\\partial v} \\cdot \\frac{\\partial v}{\\partial u} \\cdot \\frac{\\partial u}{\\partial s} \\cdot \\frac{\\partial s}{\\partial c} \\cdot \\frac{\\partial c}{\\partial a} = \\frac{\\partial f}{\\partial c} \\cdot \\frac{\\partial c}{\\partial a} = 0.2 \\cdot 1 = 0.2$'),
        ('b', '$b(w_1, x_1) = w_1 \\cdot x_1 = (-3) \\cdot (-2) = 6$', '$c(a, b) = a + b \\Rightarrow \\frac{\\partial c}{\\partial b} = 1$\n$\\frac{\\partial f}{\\partial b} = \\frac{\\partial f}{\\partial f} \\cdot \\frac{\\partial f}{\\partial t} \\cdot \\frac{\\partial t}{\\partial v} \\cdot \\frac{\\partial v}{\\partial u} \\cdot \\frac{\\partial u}{\\partial s} \\cdot \\frac{\\partial s}{\\partial c} \\cdot \\frac{\\partial c}{\\partial b} = \\frac{\\partial f}{\\partial c} \\cdot \\frac{\\partial c}{\\partial b} = 0.2 \\cdot 1 = 0.2$'),
        ('c', '$c(a, b) = a + b = -2 + 6 = 4$', '$s(c, w_2) = c + w_2 \\Rightarrow \\frac{\\partial s}{\\partial c} = 1$\n$\\frac{\\partial f}{\\partial c} = \\frac{\\partial f}{\\partial f} \\cdot \\frac{\\partial f}{\\partial t} \\cdot \\frac{\\partial t}{\\partial v} \\cdot \\frac{\\partial v}{\\partial u} \\cdot \\frac{\\partial u}{\\partial s} \\cdot \\frac{\\partial s}{\\partial c} = \\frac{\\partial f}{\\partial s} \\cdot \\frac{\\partial s}{\\partial c} = 0.2 \\cdot 1 = 0.2$'),
        ('w₂', '$w_2 = -3$ — сдвиг, значение задано.', '$s(c, w_2) = c + w_2 \\Rightarrow \\frac{\\partial s}{\\partial w_2} = 1$\n$\\frac{\\partial f}{\\partial w_2} = \\frac{\\partial f}{\\partial f} \\cdot \\frac{\\partial f}{\\partial t} \\cdot \\frac{\\partial t}{\\partial v} \\cdot \\frac{\\partial v}{\\partial u} \\cdot \\frac{\\partial u}{\\partial s} \\cdot \\frac{\\partial s}{\\partial w_2} = \\frac{\\partial f}{\\partial s} \\cdot \\frac{\\partial s}{\\partial w_2} = 0.2 \\cdot 1 = 0.2$'),
        ('s', '$s(c, w_2) = c + w_2 = 4 + (-3) = 1$', '$u(s) = -s \\Rightarrow \\frac{\\partial u}{\\partial s} = -1$\n$\\frac{\\partial f}{\\partial s} = \\frac{\\partial f}{\\partial f} \\cdot \\frac{\\partial f}{\\partial t} \\cdot \\frac{\\partial t}{\\partial v} \\cdot \\frac{\\partial v}{\\partial u} \\cdot \\frac{\\partial u}{\\partial s} = \\frac{\\partial f}{\\partial u} \\cdot \\frac{\\partial u}{\\partial s} = (-0.2) \\cdot (-1) = 0.2$'),
        ('u', '$u(s) = -s = -1$', '$v(u) = e^{u} \\Rightarrow \\frac{\\partial v}{\\partial u} = e^{u} = v = 0.37$\n$\\frac{\\partial f}{\\partial u} = \\frac{\\partial f}{\\partial f} \\cdot \\frac{\\partial f}{\\partial t} \\cdot \\frac{\\partial t}{\\partial v} \\cdot \\frac{\\partial v}{\\partial u} = \\frac{\\partial f}{\\partial v} \\cdot \\frac{\\partial v}{\\partial u} = (-0.53) \\cdot 0.37 = -0.20$'),
        ('v', '$v(u) = e^{u} = e^{-1} = 0.37$', '$t(v) = v + 1 \\Rightarrow \\frac{\\partial t}{\\partial v} = 1$\n$\\frac{\\partial f}{\\partial v} = \\frac{\\partial f}{\\partial f} \\cdot \\frac{\\partial f}{\\partial t} \\cdot \\frac{\\partial t}{\\partial v} = \\frac{\\partial f}{\\partial t} \\cdot \\frac{\\partial t}{\\partial v} = (-0.53) \\cdot 1 = -0.53$'),
        ('t', '$t(v) = v + 1 = 0.37 + 1 = 1.37$', '$f(t) = 1 / t \\Rightarrow \\frac{\\partial f}{\\partial t} = -1/t^2 = -1/1.37^2 = -0.53$\n$\\frac{\\partial f}{\\partial t} = \\frac{\\partial f}{\\partial f} \\cdot \\frac{\\partial f}{\\partial t} = 1 \\cdot (-0.53) = -0.53$'),
        ('f', '$f(t) = 1 / t = 1 / 1.37 = 0.73$', 'Начало обратного прохода:\n$\\frac{\\partial f}{\\partial f} = 1$.'),
    
    ],
    title=r"$f = \sigma(w_0 x_0 + w_1 x_1 + w_2)$:   $a = w_0 x_0$,  $b = w_1 x_1$,  $c = a + b$,  $s = c + w_2$,  $u = -s$,  $v = e^{u}$,  $t = v + 1$,  $f = 1/t$",
    size=(24, 12.5),
    panel="below",
)

Четыре правых узла — это сигмоида целиком. Ее производная выражается через ее же значение:

$$\sigma'(t) = \frac{e^{-t}}{(1 + e^{-t})^2} = \sigma(t)\,\big(1 - \sigma(t)\big) = 0.73 \cdot 0.27 = 0.20,$$

то же число, что дали четыре шага по отдельности. Поэтому в реализации сигмоиду считают одним
узлом: в обратном проходе нужен только ее выход с прямого прохода. Размер «узла» выбирают сами:
лишь бы у него была простая локальная производная.

## 4. Функции активации

Полносвязный слой — линейное отображение $h \mapsto hW + b$: растягивает, поворачивает и сдвигает
пространство признаков, но прямые оставляет прямыми. Если бы между слоями ничего не стояло,
сколько слоев ни складывай, граница между классами осталась бы гиперплоскостью — а цифры,
полумесяцы и почти любые настоящие данные ею не разделяются. **Функция активации** $\varphi$
применяется к каждому нейрону после линейного слоя и ломает линейность: $h \mapsto \varphi(hW + b)$.
Только с ней композиция слоев становится богаче одного слоя, и сеть может выучить
изогнутую границу.

**Без нелинейности слои схлопываются:**

$$(x W_1 + b_1) W_2 + b_2 = x\,(W_1 W_2) + (b_1 W_2 + b_2),$$

и сеть из любого числа слоев — та же линейная модель. Проверка на двух полумесяцах: логистическая
регрессия, сеть из двух слоев по 64 нейрона без нелинейности (`activation="identity"`), с ReLU
и с сигмоидой.

In [ ]:
X_moons, y_moons = make_moons(n_samples=500, noise=0.2, random_state=SEED)

moon_models = [
    LogisticRegression(),
    MLPClassifier(hidden_layer_sizes=(64, 64), activation="identity", solver="lbfgs", max_iter=2000,
                  random_state=SEED),
    MLPClassifier(hidden_layer_sizes=(64, 64), activation="relu", solver="lbfgs", max_iter=2000,
                  random_state=SEED),
    MLPClassifier(hidden_layer_sizes=(64, 64), activation="logistic", solver="lbfgs", max_iter=2000,
                  random_state=SEED),
]
for model in moon_models:
    model.fit(X_moons, y_moons)

plot_decision_maps(moon_models, X_moons, y_moons,
                   ["логистическая регрессия", "2 слоя без нелинейности", "2 слоя, ReLU", "2 слоя, сигмоида"])

Сеть без нелинейности проводит ту же прямую, что логистическая регрессия: 4.4 тысячи параметров
работают как три параметра прямой. С ReLU граница повторяет форму полумесяцев.

Четыре распространенные функции активации и их производные:

- **сигмоида** $\sigma(z) = \frac{1}{1 + e^{-z}}$, производная $\sigma(1 - \sigma) \le \frac{1}{4}$. При $|z| > 5$ производная почти ноль —
  нейрон **насыщен** и не учится; в глубокой сети множители $\le \frac{1}{4}$ перемножаются, и градиент
  до первых слоев не доходит — **затухание градиента**;
- **tanh** $= 2\sigma(2z) - 1$: то же насыщение, но выход симметричен вокруг нуля, производная до 1;
- **ReLU** $\max(z, 0)$: производная 1 при $z > 0$ — градиент не затухает, считается быстро; но при $z < 0$
  производная 0, и нейрон, у которого вход всегда отрицателен, перестает учиться навсегда —
  «мертвый» нейрон;
- **Leaky ReLU** $\max(z, \alpha z)$, обычно $\alpha = 0.01$ или $0.1$: при $z < 0$ производная $\alpha$, а не 0, и мертвых
  нейронов нет.

In [ ]:
alpha = 0.1
sigmoid = lambda z: 1 / (1 + np.exp(-z))

plot_activations({
    "сигмоида": (sigmoid, lambda z: sigmoid(z) * (1 - sigmoid(z))),
    "tanh": (np.tanh, lambda z: 1 - np.tanh(z) ** 2),
    "ReLU": (lambda z: np.maximum(z, 0), lambda z: (z > 0).astype(float)),
    f"Leaky ReLU, α = {alpha}": (lambda z: np.where(z > 0, z, alpha * z), lambda z: np.where(z > 0, 1.0, alpha)),
})

Сегодня в скрытых слоях почти всегда ReLU и ее варианты (Leaky ReLU, GELU); сигмоиду оставляют
на выходе, когда нужна вероятность одного класса, а софтмакс — когда классов несколько.

## 5. Софтмакс: от одного класса к десяти

Сигмоида в конце сети отвечает на вопрос «да или нет»: $\sigma(z) \in (0, 1)$ — вероятность одного
класса, вероятность другого — $1 - \sigma(z)$. Для десяти цифр нужен вектор из десяти чисел, которые
неотрицательны и в сумме дают 1. Последний линейный слой выдает десять **оценок** $z_1, \dots, z_{10}$
любого знака; **софтмакс** переводит их в распределение:

$$p_k = \frac{e^{z_k}}{\sum_{j=1}^{10} e^{z_j}}.$$

Экспонента делает все числа положительными, деление на сумму — нормирует. Самая большая оценка
дает самую большую вероятность, а разница между оценками превращается в отношение
вероятностей: $p_k / p_j = e^{z_k - z_j}$. При двух классах софтмакс — та же сигмоида разности оценок,
$p_1 = \sigma(z_1 - z_2)$, так что это ее прямое обобщение. Потеря для софтмакса — кросс-энтропия
$-\ln p_y$: минус логарифм вероятности верного класса.

**Данные.** MNIST — 70 000 рукописных цифр 28×28 пикселей. Картинка — вектор из 784 чисел,
яркость от 0 до 1; ответ — цифра от 0 до 9. Первые 50 000 картинок — обучение, следующие
10 000 — валидация, последние 10 000 — тестовый сет.

In [ ]:
mnist_X, mnist_y = fetch_openml("mnist_784", version=1, return_X_y=True, as_frame=False, parser="liac-arff")
mnist_X = (mnist_X / 255.0).astype(np.float64)       # яркость пикселя от 0 до 1
mnist_y = mnist_y.astype(int)

# стандартное деление MNIST: первые 60 000 — обучение, последние 10 000 — тестовый сет
X_train, y_train = mnist_X[:50000], mnist_y[:50000]
X_val, y_val = mnist_X[50000:60000], mnist_y[50000:60000]
X_test, y_test = mnist_X[60000:], mnist_y[60000:]

print("обучение:", X_train.shape, "валидация:", X_val.shape, "тест:", X_test.shape)

In [ ]:
show_digits(X_train[:20], y_train[:20], title="первые 20 картинок обучающей выборки: над каждой — ее класс")

Чтобы заглянуть внутрь сети, нужны ее модули — полносвязный слой, активации, софтмакс
с кросс-энтропией и контейнер `Sequential`. На семинаре их пишут руками; для лекции они лежат
в `sem10_utils.py` рядом с ноутбуком.

In [ ]:
from sem10_utils import Linear, SoftmaxCrossEntropy, make_network

Софтмакс на одной картинке. Сеть еще не обучена, веса случайные: оценки близки к нулю, и все
десять вероятностей близки к 0.1 — сеть «не знает».

In [ ]:
demo_rng = np.random.default_rng(SEED)
demo_network = make_network([256, 128, 64], activation="relu", init="he", rng=demo_rng)

example = X_train[:1]
example_label = int(y_train[0])
logits = demo_network.forward(example)[0]
probabilities = np.exp(logits - logits.max())
probabilities = probabilities / probabilities.sum()
print("верный класс:", example_label, "| ответ необученной сети:", int(np.argmax(probabilities)))
plot_softmax_bars(logits, probabilities, example_label)

## 6. Прямой проход по слоям

**Архитектура.** Полносвязная сеть 784 → 256 → 128 → 64 → 10: четыре линейных слоя, после трех
скрытых — ReLU, после последнего — софтмакс.

In [ ]:
plot_architecture([784, 256, 128, 64, 10])

Прямой проход — цепочка стадий: вход $x$ → $z_1 = x W_1 + b_1$ → $h_1 = \mathrm{ReLU}(z_1)$ → $z_2 = h_1 W_2 + b_2$ → … →
$z_4$ → $p = \mathrm{softmax}(z_4)$. На каждой стадии — вектор; его длина равна числу нейронов слоя,
в столбце показаны первые и последние шесть координат. Граф ниже интерактивный: стадии
переключаются слайдером. Оранжевый — положительные значения, синий — отрицательные; после ReLU
синих не остается.

In [ ]:
def forward_stages(network, x_row):
    """Вектор на каждой стадии прямого прохода одного объекта: вход, после каждого слоя, софтмакс."""
    stages = [("вход $x$", x_row)]
    current = x_row[None, :]
    linear_index = 0
    for layer in network.layers:
        current = layer.forward(current)
        if isinstance(layer, Linear):
            linear_index += 1
            stages.append((f"$z_{linear_index} = h_{linear_index - 1} W_{linear_index} + b_{linear_index}$", current[0]))
        else:
            stages.append((f"$h_{linear_index} = \\mathrm{{ReLU}}(z_{linear_index})$", current[0]))
    shifted = current[0] - current[0].max()
    stages.append(("$p = \\mathrm{softmax}(z_4)$", np.exp(shifted) / np.exp(shifted).sum()))
    return stages


stages = forward_stages(demo_network, X_train[0])
animate_forward_pass(stages, [784, 256, 128, 64, 10])

## 7. Инициализация весов

С какими весами начинать обучение? Сравним три варианта на сети 784 → 256 → 128 → 64 → 10
из раздела 6:

- все веса одним числом 0.01;
- случайные веса $\mathcal{N}(0, 1)$;
- инициализация He: случайные веса $\mathcal{N}(0, 2/n_{in})$, где $n_{in}$ — число входов слоя.

Через необученную сеть пропускаем 500 картинок. На каждой стадии прямого прохода считаем
типичный размер значений — корень из среднего квадрата по картинкам и нейронам,
$\sqrt{\frac{1}{n m} \sum_{i, j} h_{ij}^2}$. После одного обратного прохода смотрим на потерю и на норму
градиента по весам первого слоя; для одной картинки считаем, сколько разных значений
у 256 нейронов первого слоя.

In [ ]:
stage_names = ["вход x", "h₁", "h₂", "h₃", "оценки z₄"]
init_batch, init_labels = X_train[:500], y_train[:500]

scale_by_init = {}
max_probability_by_init = {}
init_rows = {}
for name, method in [("все веса 0.01", "constant"), ("N(0, 1)", "normal"), ("He", "he")]:
    network = make_network([256, 128, 64], activation="relu", init=method, rng=np.random.default_rng(SEED))

    stage_values = [init_batch]
    current = init_batch
    for layer in network.layers:
        current = layer.forward(current)
        is_activation = not isinstance(layer, Linear)
        is_output = layer is network.layers[-1]
        if is_activation or is_output:
            stage_values.append(current)
    scale_by_init[name] = [np.sqrt(np.mean(values ** 2)) for values in stage_values]

    loss_function = SoftmaxCrossEntropy()
    loss = loss_function.forward(current, init_labels)
    network.backward(loss_function.backward())
    max_probability_by_init[name] = loss_function.probabilities.max(axis=1)

    first_hidden = stage_values[1][0]
    init_rows[name] = {
        "разных значений у 256 нейронов h₁": len(np.unique(np.round(first_hidden, 8))),
        "потеря (кросс-энтропия)": loss,
        "норма градиента по W₁": np.linalg.norm(network.layers[0].dW),
    }

init_table = pd.DataFrame(init_rows).T
init_table["разных значений у 256 нейронов h₁"] = init_table["разных значений у 256 нейронов h₁"].astype(int)
show_table(init_table)

In [ ]:
plot_init_comparison(stage_names, scale_by_init, max_probability_by_init)

**Почему масштаб меняется от слоя к слою.** Значение нейрона следующего слоя —
$z_j = \sum_{k=1}^{n_{in}} h_k w_{kj}$. Веса независимы от входов и имеют нулевое среднее, поэтому
перекрестные слагаемые в квадрате суммы в среднем пропадают:

$$\mathbb{E} z_j^2 = \sum_{k=1}^{n_{in}} \mathbb{E} h_k^2 \cdot \mathbb{E} w_{kj}^2 = n_{in} \cdot \mathbb{E} h^2 \cdot \sigma_w^2.$$

ReLU оставляет положительную половину симметричного $z$ и обнуляет отрицательную:
$\mathbb{E}\,\mathrm{ReLU}(z)^2 = \frac{1}{2}\,\mathbb{E} z^2$. За один слой с ReLU

$$\mathbb{E} h_{new}^2 = \frac{n_{in}\,\sigma_w^2}{2} \cdot \mathbb{E} h^2,$$

то есть типичный размер умножается на $\sqrt{n_{in}\,\sigma_w^2 / 2}$.

**$\mathcal{N}(0, 1)$: взрыв масштаба.** При $\sigma_w = 1$ множитель равен $\sqrt{n_{in}/2}$: $\sqrt{784/2} \approx 20$ на первом
слое, $\sqrt{256/2} \approx 11$ на втором, $\sqrt{128/2} = 8$ на третьем. Так и выходит на графике: 0.33 на входе,
6 после первого слоя, 75, 556 и около 3780 на оценках. Софтмакс от оценок порядка тысяч отдает
почти всю вероятность одному классу: на всех 500 картинках наибольшая вероятность около 1.
Необученная сеть уверена и почти всегда ошибается: потеря около 23 вместо $\ln 10 \approx 2.3$, градиент
по $W_1$ в тысячу раз больше, чем у He, и шаг градиентного спуска с обычным $\eta$ выбрасывает
веса еще дальше.

**Все веса 0.01: нулевой градиент.** У всех 256 нейронов первого слоя одно значение: одинаковые
веса дают одинаковую взвешенную сумму пикселей. Так же одинаковы нейроны следующих слоев
и все десять оценок; софтмакс дает каждому классу ровно 0.1, потеря — $\ln 10$. Градиент по оценкам
$G = p - Y$ в сумме по классам равен нулю: $\sum_k (p_k - Y_k) = 1 - 1 = 0$. Последний слой
передает назад $G W_4^\top$, а при одинаковых весах $0.01$ это $0.01 \sum_k G_{ik} = 0$: градиент по всем
слоям, кроме последнего, — ноль. В таблице 0.0000; точное значение порядка $10^{-17}$ — ошибка
округления. Даже когда он не ноль, одинаковые
нейроны получают одинаковые градиенты и после шага остаются одинаковыми: слой
из 256 нейронов работает как один. Эту **симметрию** ломает только случайная инициализация.

**He: масштаб сохраняется.** Множитель $\sqrt{n_{in}\,\sigma_w^2 / 2}$ равен единице при $\sigma_w^2 = 2 / n_{in}$ — это
и есть инициализация He. Размер значений на всех стадиях около 0.33, как у входа; наибольшая
вероятность — от 0.1 до 0.3: до обучения сеть и не должна быть уверена. Потеря 2.33 — около
$\ln 10$, градиент порядка единицы. Для сигмоиды и tanh тот же расчет без двойки от ReLU
дает инициализацию Xavier, $\mathcal{N}(0, 2/(n_{in} + n_{out}))$.

## 8. Сеть для MNIST: обучение

Та же архитектура 784 → 256 → 128 → 64 → 10 встроенными средствами sklearn. `MLPClassifier`
сам строит слои по `hidden_layer_sizes`, ставит ReLU между ними и софтмакс с кросс-энтропией
в конце; инициализация — Xavier.

**Обучение.** Потеря — кросс-энтропия софтмакса. Обратный проход дает ее градиент по всем
242 762 весам, шаг градиентного спуска сдвигает веса против градиента. Градиент считают
не по всей выборке, а по мини-батчу из 64 случайных картинок: шаг дешевле, направление
в среднем то же. **Эпоха** — один проход по всем 50 000 картинкам, около 780 шагов.
Обучаем по одной эпохе за вызов, чтобы увидеть, как растет качество.

In [ ]:
mnist_network = MLPClassifier(hidden_layer_sizes=(256, 128, 64), activation="relu", solver="sgd", learning_rate_init=0.1,
                              batch_size=64, momentum=0.0, max_iter=1, warm_start=True, random_state=SEED)

history = {"потеря на обучении": [], "accuracy на обучении": [], "accuracy на валидации": []}
for epoch in range(10):
    mnist_network.fit(X_train, y_train)
    history["потеря на обучении"].append(mnist_network.loss_)
    history["accuracy на обучении"].append(mnist_network.score(X_train, y_train))
    history["accuracy на валидации"].append(mnist_network.score(X_val, y_val))
    print(f"эпоха {epoch + 1:2d}: потеря {mnist_network.loss_:.4f}, accuracy на обучении "
          f"{history['accuracy на обучении'][-1]:.4f}, на валидации {history['accuracy на валидации'][-1]:.4f}")

clean_accuracy = mnist_network.score(X_test, y_test)
print(f"accuracy на тесте после 10 эпох: {clean_accuracy:.4f}")
plot_histories({"784 → 256 → 128 → 64 → 10": history}, accuracy_ylim=(0.9, 1.0))

Потеря на обучении падает каждую эпоху, accuracy на валидации растет и к десятой эпохе
почти перестает меняться. Сеть ошибается примерно на 2–3% тестовых картинок; логистическая
регрессия на тех же пикселях — на 7%. Предсказания на тестовых картинках:

In [ ]:
test_predicted = mnist_network.predict(X_test[:20])
show_digits(X_test[:20], y_test[:20], test_predicted, title="тестовые картинки: класс → предсказание сети")

## 9. Производные по входу

До сих пор градиент считали по весам. Тот же обратный проход дает и производную по входу:
это градиент, который первый слой отдает «наружу», $\frac{\partial L}{\partial x} = G_1 W_1^\top$. Обученная сеть — функция
пикселей, и можно спросить, какие пиксели и как надо изменить, чтобы ответ стал другим.

Градиент кросс-энтропии по входу считаем руками тем же обратным проходом: веса $W_1, W_2$ и сдвиги
берем из обученной модели (`coefs_`, `intercepts_`).

In [ ]:
def forward_with_masks(network, X):
    """Прямой проход по весам sklearn; возвращает оценки и маски ReLU каждого скрытого слоя."""
    current = X
    masks = []
    for W, b in zip(network.coefs_[:-1], network.intercepts_[:-1]):
        pre_activation = current @ W + b
        masks.append(pre_activation > 0)
        current = np.maximum(pre_activation, 0)
    logits = current @ network.coefs_[-1] + network.intercepts_[-1]
    return logits, masks


def backward_to_input(network, grad_logits, masks):
    """Обратный проход от градиента по оценкам до градиента по входу."""
    grad = grad_logits @ network.coefs_[-1].T
    for W, mask in zip(reversed(network.coefs_[:-1]), reversed(masks)):
        grad = (grad * mask) @ W.T
    return grad


def input_gradient(network, X, y):
    """Градиент средней кросс-энтропии по входу: обратный проход до картинки."""
    logits, masks = forward_with_masks(network, X)
    shifted = logits - logits.max(axis=1, keepdims=True)
    probabilities = np.exp(shifted) / np.exp(shifted).sum(axis=1, keepdims=True)
    grad_logits = probabilities.copy()
    grad_logits[np.arange(len(y)), y] -= 1.0              # dL/dlogits = p − Y для каждого объекта
    grad_input = backward_to_input(network, grad_logits, masks)
    return grad_input

**Атака на сеть** (adversarial attack). Метод быстрого знака градиента (fast gradient sign method, FGSM;
Goodfellow и др., 2014) сдвигает каждый пиксель на $\varepsilon$ туда, где ошибка растет:

$$x' = x + \varepsilon\, \operatorname{sign}\Big(\frac{\partial L}{\partial x}\Big).$$

Картинка меняется на $\varepsilon$ в каждом пикселе — для человека почти незаметно, а сеть ошибается.

In [ ]:
epsilon = 0.1
examples = X_test[:5]
example_labels = y_test[:5]

gradient = input_gradient(mnist_network, examples, example_labels)
perturbation = epsilon * np.sign(gradient)
attacked = np.clip(examples + perturbation, 0, 1)

show_adversarial(examples, perturbation, attacked, example_labels,
                 mnist_network.predict(examples), mnist_network.predict(attacked), epsilon)

In [ ]:
epsilons = [0.0, 0.02, 0.05, 0.1, 0.15, 0.2]
attacked_accuracy = []
test_gradient = input_gradient(mnist_network, X_test, y_test)
for value in epsilons:
    attacked_test = np.clip(X_test + value * np.sign(test_gradient), 0, 1)
    attacked_accuracy.append(mnist_network.score(attacked_test, y_test))

plot_curve(epsilons, attacked_accuracy, "ε — сдвиг каждого пикселя", "accuracy на атакованном тесте",
           title="FGSM: сеть 784 → 256 → 128 → 64 → 10 на MNIST")
for value, score in zip(epsilons, attacked_accuracy):
    print(f"ε = {value:.2f}: accuracy {score:.3f}")

**Карта важности пикселей** (saliency map): модуль производной оценки верного класса по каждому
пикселю, $\big|\frac{\partial z_y}{\partial x_{ij}}\big|$. Яркие пиксели сильнее всего меняют оценку класса. Сравним
карты для исходных картинок и для тех же картинок после атаки.

In [ ]:
def class_score_gradient(network, X, y):
    """Градиент оценки верного класса (логита) по входу."""
    _, masks = forward_with_masks(network, X)
    grad_logits = np.zeros((len(y), network.coefs_[-1].shape[1]))
    grad_logits[np.arange(len(y)), y] = 1.0           # d logit_y / d logits — единица в верном классе
    grad_input = backward_to_input(network, grad_logits, masks)
    return grad_input


saliency_clean = np.abs(class_score_gradient(mnist_network, examples, example_labels))
saliency_attacked = np.abs(class_score_gradient(mnist_network, attacked, example_labels))
show_saliency_pairs(examples, attacked, saliency_clean, saliency_attacked, example_labels,
                    mnist_network.predict(examples), mnist_network.predict(attacked))

Карты шумные и до атаки, и после: важные пиксели не выстраиваются вдоль штрихов, а рассыпаны
по всему полю, в том числе по фону, где чернил нет. Полносвязная сеть не знает, что соседние
пиксели связаны, — каждый вес смотрит на один пиксель независимо, и на фон она опирается
не меньше, чем на штрих. Поэтому ее и удается сбить сдвигом каждого пикселя на $\varepsilon$:
атака меняет как раз те фоновые пиксели, на которые сеть зря смотрит. Сверточные сети
следующего занятия устроены иначе — их карты важности повторяют форму цифры.

Где еще нужна производная по входу:

- **защита от атак** — обучение на атакованных примерах (adversarial training);
- **объяснение ответа** — карты важности, интегрированные градиенты;
- **визуализация признаков** — градиентный подъем по входу, чтобы найти картинку, на которую
  сильнее всего реагирует нейрон (DeepDream);
- **перенос стиля** — оптимизация картинки, а не весов: веса обученной сети фиксированы, по входу
  ищут картинку с содержанием одной и стилем другой (Gatys и др., 2015).

## 10. Производные по правильному ответу

Третий аргумент потери — ответ $y$. Для кросс-энтропии с софтмаксом $L = -\sum_k y_k \ln p_k$, и при мягких
ответах $y_k \in [0, 1]$ производная $\frac{\partial L}{\partial y_k} = -\ln p_k$ — «насколько дорого сеть платит за класс $k$».
Когда метки не данность, а параметры, их тоже учат градиентным спуском:

- **исправление ошибок разметки** (meta label correction): метки шумных объектов заменяют обучаемыми
  мягкими метками и подбирают их так, чтобы сеть, обученная на них, лучше работала на чистой
  валидации;
- **дистилляция датасета** (dataset distillation): ищут маленький синтетический набор — и картинки,
  и мягкие метки, — обучение на котором дает почти ту же сеть, что на полном наборе;
- в **дистилляции знаний** (knowledge distillation) ответ $y$ для маленькой сети выдает большая — мягкие
  вероятности учителя; а в **GAN** (generative adversarial network) генератор учится через градиент
  дискриминатора по его входу — сгенерированной картинке.

## Итоги

- сеть — функция $L(f(x; W), y)$ входа, весов и ответа; обучение — градиентный спуск по $W$;
- обратное распространение ошибки — цепное правило на вычислительном графе: каждый узел
  умножает пришедший справа градиент на свою локальную производную;
- софтмакс переводит десять оценок в вероятности; при двух классах это сигмоида разности оценок;
- веса инициализируют случайно, иначе нейроны слоя одинаковы и градиент по ранним слоям нулевой;
  масштаб $\sqrt{2 / n_{in}}$ (He) сохраняет размер значений от слоя к слою, $\mathcal{N}(0, 1)$ его взрывает;
- без нелинейности сеть линейна; сигмоида насыщается, ReLU — нет, но может «умереть»,
  Leaky ReLU этого избегает;
- тот же обратный проход дает производные по входу — атаки, карты важности, перенос стиля —
  и по ответу — обучаемые метки.